In [31]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from sklearn.cluster import KMeans

In [32]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)

    for node in bn.nodes:
        cpd = bn.cpds[node]
        
        probs = bn.predict_probability(data, node)
        
        true_values = data[node]
        
        y_dummies = pd.get_dummies(true_values)
        
        y_dummies = y_dummies.reindex(columns=probs.columns, fill_value=0)
        
        p_actual = (probs * y_dummies).sum(axis=1)
        
        log_likelihood += np.sum(np.log(p_actual + 1e-9))

        num_parent_configs = cpd.shape[0]
        num_node_states = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)

    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [33]:
from sklearn.mixture import GaussianMixture # <--- NEW IMPORT
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [34]:
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_bn_no_context.csv")

df_pl = df_pl.drop(["Unnamed: 0"], axis = 1)

df_pl

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,31201.797665,29.28,67.985714,3.630000,2.570000,21.39,6.903,-3.803166,18.166685,-1.872662,5.83,-0.3,17.3,0.000009,62.749213,36.747184
1,33846.381917,28.82,67.985714,3.690000,1.910000,17.10,7.686,-5.237136,18.626276,-3.397579,2.72,0.5,16.9,0.000009,65.250459,57.512070
2,36057.305377,29.96,62.600000,3.740000,2.060000,20.47,8.676,-0.960496,18.707747,-6.157262,3.18,0.8,17.1,0.000008,75.989515,135.142137
3,37895.481599,29.70,62.200000,3.650000,2.230000,23.58,9.281,-2.441439,19.149239,-6.142614,3.94,1.0,17.6,0.000009,84.200841,180.098848
4,41087.789260,30.39,62.500000,3.650000,2.330000,25.53,10.337,-2.287307,18.077861,-4.112383,3.57,0.0,17.7,0.000009,101.918024,247.386988
5,42619.735974,30.23,62.900000,3.810000,2.620000,26.31,10.955,-0.784100,18.008657,-3.437896,1.53,0.0,17.1,0.000008,115.869679,265.323092
6,43053.616233,28.68,63.200000,3.910000,3.050000,27.16,11.452,1.117069,18.330801,-4.023830,0.26,-1.2,17.3,0.000008,111.382755,275.386821
7,44990.974498,29.94,64.900000,4.020000,3.550000,23.73,11.605,0.407324,18.238027,-3.367862,3.85,-0.3,17.0,0.000008,103.683241,307.471216
8,47637.139301,30.99,66.300000,4.070000,3.810000,20.68,11.881,2.067846,17.905031,-2.532527,3.30,-1.0,17.6,0.000008,123.085022,169.995130
9,49142.895037,30.67,68.200000,4.300000,4.000000,17.59,11.396,2.946955,17.821550,-2.650770,3.82,0.2,17.3,0.000008,112.220174,76.266802


In [35]:
pl = pd.DataFrame()
pl = df_pl.copy()
pl['VABI_1'] = pl['VABI'].shift(1)
pl['FDI_3'] = pl['FDI'].shift(3)

In [36]:
pl = pl.dropna()

In [37]:
discretised_pl = pd.DataFrame(index=pl.index)

for col in pl.columns:
    no_unique = pl[col].nunique()
    
    if no_unique <= 1:
        discretised_pl[col] = 0
    else:
        try:
            discretised_pl[col] = pd.qcut(
                pl[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_pl[col] = pl[col].rank(method='plnse').astype(int) - 1
    num_states = discretised_pl[col].nunique()
    
    if num_states == 2:
        discretised_pl[col] = discretised_pl[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_pl[col] = discretised_pl[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_pl[col] = discretised_pl[col].astype(str)

print("Discretised data:")
print(discretised_pl)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
3    Low  High   Low   Low   Low  High   Low   Low  High   Low  High  High   
4    Low  High   Low   Low   Low  High   Low   Low   Low   Low  High  High   
5    Low  High   Low   Low   Low  High   Low   Low   Low   Low   Low  High   
6    Low   Low   Low   Low   Low  High   Low   Low  High   Low   Low   Low   
7    Low  High   Low   Low  High  High   Low   Low  High   Low  High   Low   
8    Low  High   Low  High  High  High  High  High   Low  High   Low   Low   
9    Low  High   Low  High  High   Low   Low  High   Low  High  High  High   
10  High   Low  High  High  High   Low   Low  High   Low  High   Low  High   
11  High   Low  High  High  High   Low  High   Low   Low  High   Low  High   
12  High   Low  High  High  High   Low  High  High   Low  High   Low   Low   
13  High   Low  High   Low   Low   Low  High  High  High   Low   Low   Low   
14  High   Low  High   Low   Low   Low  High  

In [38]:
sm = StructureModel()

In [39]:
sm.add_edges_from([  
  ('EUPC', 'REST'),
    ('EUPC', 'ARP'),
    ('EUPC', 'PAM'),
    ('VABI_1', 'EUPC'),
    ('FDI_3', 'PAM')
])



In [40]:
sm.edges

OutEdgeView([('EUPC', 'REST'), ('EUPC', 'ARP'), ('EUPC', 'PAM'), ('VABI_1', 'EUPC'), ('FDI_3', 'PAM')])

In [41]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [43]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_pl)

bn.fit_cpds(discretised_pl, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_pl)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 3263.61 (LogLikelihood: -1616.41)


In [45]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'REST'],
    discretised_data=discretised_pl,
    raw_data=pl
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'REST']
  - Removing direct edge: EUPC -> REST
LV Model BIC: 3812.67 (LogLikelihood: -1885.82)


In [46]:
bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'ARP'], 
    discretised_data=discretised_pl,
    raw_data=pl
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'ARP']
  - Removing direct edge: EUPC -> ARP
LV Model BIC: 3812.67 (LogLikelihood: -1885.82)


In [48]:
bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'PAM'], 
    discretised_data=discretised_pl,
    raw_data=pl
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'PAM']
  - Removing direct edge: EUPC -> PAM
LV Model BIC: 3812.67 (LogLikelihood: -1885.82)


In [50]:
bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['VABI_1', 'EUPC'], 
    discretised_data=discretised_pl,
    raw_data=pl
)

Estimating LV based on continuous GMM distributions in: ['VABI_1', 'EUPC']
  - Removing direct edge: VABI_1 -> EUPC
LV Model BIC: 3807.54 (LogLikelihood: -1885.82)
